# ▶ 00 Notebook Driver

|                  |                                                                                             |
| ---------------- | ------------------------------------------------------------------------------------------- |
| Title            | 00 Notebook Driver                                                                          |
| Case Study       | Climate-Adjusted Mortality for EU Long-Term Liabilities                                     |
| Research Project | CC230 Quantifying Mortality and Morbidity Impacts from Climate Risks                       |
| Notebook         | `00_notebook_driver.ipynb`                                                                  |
| Authors          | Carlos Arocha, FSA and Jêrôme Crugnola-Humbert                                              |
| Sponsor          | Catastrophe and Climate Strategic Research Program, Society of Actuaries Research Institute |
| Date             | `2026-05-11`                                                                                |
| Version          | `v1.0.0`                                                                                    |
| Questions?       | Questions regarding this notebook should be directed to the SOA project contact             |

---

## Purpose
Optional driver to run the notebook suite sequentially. The user has the ability to set `PAUSE_FOR_REVIEW = True` if a pause after each run is needed for review before contining. Set it to `False` fpor a fully automated run.

---

## Inputs
`None`

---

## Outputs
Report file
- `CC230 Toolkit/reports/{NOTEBOOK}.html`

---

## Dependencies

This notebook can be run in Google Colab or any compatible Jupyter environment, such as:

* JupyterLab
* Jupyter Notebook
* VS Code with the Jupyter extension
* Anaconda Navigator
* Kaggle Notebooks

The notebook was developed using:

* Python 3.14.5
* Required libraries installed through the setup cell below

---

## Disclaimer
The Jupyter notebooks available through this repository are provided for research, educational, and illustrative purposes only. They are not intended to constitute actuarial advice or to represent production-ready systems without further validation, customization, and governance by the user.

While reasonable efforts have been made to ensure accuracy and consistency, no guarantee is given as to completeness, fitness for a particular purpose, or compliance with applicable actuarial standards or regulatory requirements.

Users are responsible for assessing the suitability of the materials for their specific use cases and for implementing appropriate controls, validation, and documentation prior to any operational use.

---

## License
© 2026 Society of Actuaries. Except where otherwise noted, the textual content of this book is licensed under the Creative Commons Attribution–Noncommercial–Share Alike 4.0 International License (CC BY-NC-SA 4.0). Source code samples, including scripts, notebooks, and code blocks where indicated, are licensed under the MIT License, unless otherwise stated.

---

## Setup

In [1]:
# Setup

from pathlib import Path, PureWindowsPath
import time
import subprocess
import sys

from datetime import datetime
from IPython.display import display, HTML

# Set this to False to run all notebooks without manual review pauses
PAUSE_FOR_REVIEW = False

# If True, the driver stops as soon as one notebook fails
# If False, it records the failure and continues to the next notebook
STOP_ON_ERROR = True

NOTEBOOK_DIR = Path.cwd()

EXECUTED_OUTPUT_DIR = NOTEBOOK_DIR / "executed_notebooks"
EXECUTED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOKS = [
    "01_data_ingestion_and_preparation.ipynb",
    "02_climate_features.ipynb",
    "03_baseline_mortality_model.ipynb",
    "04_climate_calibration.ipynb",
    "05_projection_engine.ipynb",
    "06_sensitivity_and_validation.ipynb"
]

NOTEBOOK_FILE = Path().cwd().parent / "notebooks" / "00_notebook_driver.ipynb"    

REPORTS_PATH = Path().cwd().parent / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("Notebook driver initialized.")
print(f"Notebook folder: {PureWindowsPath(NOTEBOOK_DIR).name}")
print(f"Executed notebook output folder: {PureWindowsPath(EXECUTED_OUTPUT_DIR).name}")
print(f"Pause for review: {PAUSE_FOR_REVIEW}")


Notebook driver initialized.
Notebook folder: notebooks
Executed notebook output folder: executed_notebooks
Pause for review: False


## Settings

In [2]:
# Settings for notebook execution

# Set this to False to run all notebooks without manual review pauses
PAUSE_FOR_REVIEW = True

# If True, the driver stops as soon as one notebook fails
# If False, it records the failure and continues to the next notebook
STOP_ON_ERROR = True

## Validation

In [3]:
# Validate that all required notebooks exist in the notebook directory

missing_notebooks = [
    notebook_name
    for notebook_name in NOTEBOOKS
    if not (NOTEBOOK_DIR / notebook_name).exists()
]

if missing_notebooks:
    missing_text = "\n".join(f"  - {name}" for name in missing_notebooks)
    raise FileNotFoundError(
        "The following notebooks were not found in NOTEBOOK_DIR:\n"
        f"{missing_text}\n\n"
        "Move this driver notebook into the same folder as notebooks 1 to 6, "
        "or update NOTEBOOK_DIR."
    )

print("All notebooks found. Ready to run!")


All notebooks found. Ready to run!


## Helper Functions

In [4]:
# Helper functions

def execute_notebook(notebook_name: str) -> dict:
    """
    Execute one notebook using nbconvert and save an executed copy.

    Returns a dictionary with status information.
    """
    input_path = NOTEBOOK_DIR / notebook_name
    output_name = notebook_name.replace(".ipynb", "_executed.ipynb")
    output_path = EXECUTED_OUTPUT_DIR / output_name

    print("=" * 90)
    print(f"Starting: {notebook_name}")
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Executed copy will be saved to: {PureWindowsPath(output_path).name}")
    print("=" * 90)

    start_time = time.time()

    command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(input_path),
        "--output",
        output_name,
        "--output-dir",
        str(EXECUTED_OUTPUT_DIR),
        "--ExecutePreprocessor.timeout=-1",
        "--ExecutePreprocessor.kernel_name=python3",
    ]

    completed = subprocess.run(
        command,
        cwd=str(NOTEBOOK_DIR),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    elapsed_seconds = time.time() - start_time

    print(completed.stdout)

    status = "success" if completed.returncode == 0 else "failed"

    result = {
        "notebook": notebook_name,
        "status": status,
        "return_code": completed.returncode,
        "elapsed_seconds": round(elapsed_seconds, 2),
        "executed_output": str(output_path),
        "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    if status == "success":
        print(f"Completed successfully: {notebook_name}")
    else:
        print(f"Failed: {notebook_name}")

    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print()

    return result

def pause_for_review_if_requested(notebook_name: str, is_last_notebook: bool) -> None:
    """
    Pause between notebooks when PAUSE_FOR_REVIEW is True.
    """
    if not PAUSE_FOR_REVIEW or is_last_notebook:
        return

    display(HTML(
        f"""
        <div style="
            border: 3px solid #d97706;
            background-color: #fff7ed;
            padding: 18px;
            border-radius: 8px;
            font-size: 16px;
            margin: 12px 0;
        ">
            <h2 style="margin-top: 0;">⚠️ Review Required</h2>
            <p><strong>Notebook:</strong> <code>{notebook_name}</code></p>
            <p>Review the output before continuing.</p>
            <p>Press <strong>Enter</strong> to continue to the next notebook, or type <strong>q</strong> to quit.</p>
        </div>
        """
    ))

    response = input("👉👉👉 Your choice [Enter = continue, q = quit]: ").strip().lower()

    if response in {"q", "quit", "stop", "exit"}:
        raise KeyboardInterrupt("Notebook driver stopped by user during review pause")

## Run Notebooks

In [5]:
# Run notebooks sequentially

run_results = []

driver_start = time.time()

for index, notebook_name in enumerate(NOTEBOOKS, start=1):
    is_last_notebook = index == len(NOTEBOOKS)

    result = execute_notebook(notebook_name)
    run_results.append(result)

    if result["status"] != "success" and STOP_ON_ERROR:
        raise RuntimeError(
            f"Stopping because notebook failed: {notebook_name}. "
            "Set STOP_ON_ERROR = False if you want the driver to continue after failures."
        )

    pause_for_review_if_requested(
        notebook_name=notebook_name,
        is_last_notebook=is_last_notebook,
    )

total_elapsed_seconds = round(time.time() - driver_start, 2)

print("=" * 90)
print("Driver run complete.")
print(f"Total elapsed seconds: {total_elapsed_seconds}")
print("=" * 90)

Starting: 01_data_ingestion_and_preparation.ipynb
Started at: 2026-07-21 08:54:05
Executed copy will be saved to: 01_data_ingestion_and_preparation_executed.ipynb
[NbConvertApp] Converting notebook c:\Users\ca\OneDrive\Documents\@ PROJECTS\CC230 Toolkit\notebooks\01_data_ingestion_and_preparation.ipynb to notebook
C:\venvs\CC230\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
Traceback (most recent call last):
  File "<frozen runpy

RuntimeError: Stopping because notebook failed: 01_data_ingestion_and_preparation.ipynb. Set STOP_ON_ERROR = False if you want the driver to continue after failures.

In [ ]:
# Create report
subprocess.run(
    [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        str(NOTEBOOK_FILE),
        "--to",
        "html",
        "--no-input",
        "--output-dir",
        str(REPORTS_PATH),
    ],
    check=True,
)

CompletedProcess(args=['c:\\Users\\ca\\OneDrive\\Documents\\@ PROJECTS\\CC230 Toolkit\\.venv\\Scripts\\python.exe', '-m', 'jupyter', 'nbconvert', 'c:\\Users\\ca\\OneDrive\\Documents\\@ PROJECTS\\CC230 Toolkit\\notebooks\\00_notebook_driver.ipynb', '--to', 'html', '--no-input', '--output-dir', 'c:\\Users\\ca\\OneDrive\\Documents\\@ PROJECTS\\CC230 Toolkit\\reports'], returncode=0)